# Bias Mitigation — Calibrated Equalized Odds (post-processing)

**AIF360 class:** `aif360.algorithms.postprocessing.CalibratedEqOddsPostprocessing` &nbsp;|&nbsp; requires `pip install -e .[mitigation]`

Equalizes error rates while preserving calibration of the predicted probabilities.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from fair_ed.mitigation.base import (
    load_splits, group_dicts, to_aif360_dataset,
    dataset_from_predictions, evaluate_subgroups,
)
from fair_ed.modeling import prepare_xy, standardize, FEATURE_COLUMNS

# --- Prediction task and binary protected attribute (edit to taste) ---
OUTCOME = "outcome_hospitalization"   # e.g. outcome_critical, outcome_sepsis, outcome_pe, ...
PROTECTED = "sex_priv"                 # 1 = male (privileged), 0 = female (unprivileged)

df_train, df_test = load_splits()
for _df in (df_train, df_test):
    _df[PROTECTED] = (_df["gender"].astype(str).str.upper().str[0] == "M").astype(int)

X_train, X_test, y_train, y_test = prepare_xy(df_train, df_test, OUTCOME)
X_train, X_test, _ = standardize(X_train, X_test)
FEAT = list(X_train.columns)

train_frame = X_train.assign(**{PROTECTED: df_train[PROTECTED].to_numpy(), OUTCOME: y_train.to_numpy()})
test_frame = X_test.assign(**{PROTECTED: df_test[PROTECTED].to_numpy(), OUTCOME: y_test.to_numpy()})

priv, unpriv = group_dicts(PROTECTED)

def subgroup_labels(ds):
    return np.where(ds.protected_attributes.ravel() == 1, "Male", "Female")

## Apply Calibrated Equalized Odds and evaluate subgroups

In [ ]:
from sklearn.model_selection import train_test_split
from fair_ed.mitigation.postprocessing import calibrated_equalized_odds

# Post-processing methods fit on a held-out validation slice of the training set
# (true labels + the base model's predictions), then transform test predictions.
tr_frame, va_frame = train_test_split(
    train_frame, test_size=0.3, random_state=42, stratify=train_frame[OUTCOME]
)
tr_ds = to_aif360_dataset(tr_frame, FEAT, OUTCOME, PROTECTED)
va_ds = to_aif360_dataset(va_frame, FEAT, OUTCOME, PROTECTED)
test_ds = to_aif360_dataset(test_frame, FEAT, OUTCOME, PROTECTED)

clf = LogisticRegression(max_iter=1000).fit(tr_ds.features, tr_ds.labels.ravel())

def predict_ds(ds):
    score = clf.predict_proba(ds.features)[:, 1]
    return dataset_from_predictions(ds, (score >= 0.5).astype(int), score)

va_pred, test_pred = predict_ds(va_ds), predict_ds(test_ds)

out = calibrated_equalized_odds(va_ds, va_pred, test_pred, priv, unpriv, cost_constraint="weighted")

y_pred = out.labels.ravel()
y_score = test_pred.scores.ravel()  # AUC uses the original (un-thresholded) scores
ci = evaluate_subgroups(test_ds.labels.ravel(), y_pred, y_score, subgroup_labels(test_ds), B=200)
print(ci.pivot(index="Group", columns="Metric", values="Mean").round(3))